# 국회도서관 학술논문 검색 기능 검증 (v2.3.1)

- Jupyter Notebook의 작업 폴더를 프로그램 코드 폴더(`graduate_paper_lab`)로 맞추세요.
- 이 실습은 **가상의 API 응답**을 사용하므로 비밀키나 인터넷 접속이 필요하지 않습니다.
- 각 기능 셀 다음에는 별도의 **assert 검증 셀**이 있습니다.

In [ ]:
# 1. 공통 모듈 준비 (사용 중인 작업 폴더가 프로젝트 최상위여야 합니다.)
from pathlib import Path
from io import BytesIO
from xml.sax.saxutils import escape
import pandas as pd
from paperlab import parse_search_xml, filter_paper_records, missing_title_rows, search_papers, API_DETAIL_URL
from nanet_detail import parse_nanet_detail_html
from citation_styles import suggested_citation, format_reference
print("작업 위치:", Path.cwd())

## 실습 1 — 학술논문과 일반도서 구분

In [ ]:
# API에서 다양한 자료가 섞여 왔다는 상황을 가정합니다.
raw_df = pd.DataFrame([
    {"controlno":"KINX2026064882", "DB":"국내학술기사", "자료명/저자사항":"애플 Siri 서비스의 포렌식 분석 연구"},
    {"controlno":"TG2026000813", "DB":"일반도서", "자료명/저자사항":"(교과서가 쉬워지는) 초등 필수 백과"},
    {"controlno":"KDMT00001", "DB":"학위논문", "자료명/저자사항":"RDS 포렌식 조사 연구"},
])
only_papers = filter_paper_records(raw_df)
print(only_papers[["DB","자료명/저자사항"]].to_string(index=False))

In [ ]:
# ✅ 실습 1 검증 셀
assert len(raw_df) == 3
assert len(only_papers) == 2
assert "일반도서" not in only_papers["DB"].tolist()
assert "학위논문" in only_papers["DB"].tolist()
print("실습 1 통과: 일반도서는 제외되고 논문 자료만 남았습니다.")

## 실습 2 — 제목 누락 상태 그대로 확인

In [ ]:
# 국회도서관 원문에는 제목이 있어도 API에는 제목이 빠질 수 있습니다.
xml = """<response><header><resultCode>00</resultCode></header><total>2</total>
<record><item><name>controlno</name><value>KINX2026064882</value></item>
<item><name>DB</name><value>국내학술기사</value></item></record>
<record><item><name>controlno</name><value>KINX2026064883</value></item>
<item><name>자료명/저자사항</name><value>포렌식 조사 절차 연구</value></item>
<item><name>DB</name><value>국내학술기사</value></item></record></response>""".encode('utf-8')
count,records=parse_search_xml(xml)
df=pd.DataFrame(records).fillna('')
missing = missing_title_rows(df)
print(df.to_string(index=False)); print("제목 없는 건수:",len(missing))

In [ ]:
# ✅ 실습 2 검증 셀
assert count == 2
assert len(missing) == 1
assert missing.iloc[0]["controlno"] == "KINX2026064882"
assert missing.iloc[0]["자료명/저자사항"] == ""
print("실습 2 통과: 누락 제목을 지어내지 않고 분리했습니다.")

## 실습 3 — 공식 상세검색 엔드포인트 사용

In [ ]:
# 외부 인터넷 대신 간단한 가짜 응답 객체를 사용합니다.
class FakeResponse:
    status_code=200
    content=xml
    def raise_for_status(self):
        pass
class FakeSession:
    def __init__(self):
        self.params=[]
    def get(self,url,params,timeout):
        self.params.append((url, params))
        return FakeResponse()
fake=FakeSession()
_,data=search_papers('검증용키','포렌식',pages=1,db_name='국내학술기사',session=fake)
print("검색 엔드포인트:", fake.params[0][0])
print("자료 유형 인자:", fake.params[0][1].get('dbname'))

In [ ]:
# ✅ 실습 3 검증 셀
assert fake.params[0][0] == API_DETAIL_URL
assert fake.params[0][1]['dbname'] == '국내학술기사'
assert len(data) == 2
print("실습 3 통과: 논문 자료유형을 서버에서 지정합니다.")

## 실습 4 — 실제 상세화면 구조를 본뜬 HTML에서 확인된 정보 추출

In [ ]:
# 사용자에게 보인 상세페이지의 글자만을 예제로 작성했습니다. (실제 사이트 호출 아님)
page = """<html><body><h3>애플 Siri 서비스의 포렌식 분석 연구</h3>
<dl><dt>저자명</dt><dd>조현지, 양지연, 장진수</dd>
<dt>발행사항</dt><dd>서울 : 한국정보보호학회, 2026. 4. 30</dd>
<dt>수록지명</dt><dd>정보보호학회논문지 제36권 제2호 (2026년 4월), p. 611-622</dd></dl></body></html>"""
detail = parse_nanet_detail_html(page)
print(detail)

In [ ]:
# ✅ 실습 4 검증 셀
assert detail['자료명/저자사항'].startswith('애플 Siri 서비스')
assert '조현지' in detail['저자']
assert detail['발행년'] == '2026'
assert 'p. 611-622' in detail['수록지명']
print("실습 4 통과: 실제 HTML에 표시된 서지정보만 추출했습니다.")

## 실습 5 — 정식 참고문헌 형식 검증

In [ ]:
paper={**detail, 'DB':'국내학술기사'}
reference = format_reference(suggested_citation(paper))
print(reference)

In [ ]:
# ✅ 실습 5 검증 셀
assert '“애플 Siri 서비스의 포렌식 분석 연구”,' in reference
assert '제36권' in reference and '제2호' in reference
assert '611-622쪽' in reference
assert '2026년 4월 30일' in reference
assert '[DOI 미확인]' in reference
print("실습 5 통과: 확인된 정보로 정식 서지를 작성하고 DOI는 미확인 처리했습니다.")

## 실습 6 — 대학별 학위논문 별도 메뉴와 중복 제거

In [ ]:
# Streamlit을 실행하지 않아도 메뉴의 코드 배치를 확인할 수 있습니다.
app_source=(Path.cwd() / 'app.py').read_text(encoding='utf-8')
legacy_source=(Path.cwd() / 'legacy_views_v22.py').read_text(encoding='utf-8')
print('분리 메뉴:', '대학 학위논문 검색' in app_source)

In [ ]:
# ✅ 실습 6 검증 셀
assert "'국회도서관 논문 검색', '대학 학위논문 검색'" in app_source
assert 'def render_university_theses():' in legacy_source
assert 'with st.expander("🏛 대학별 논문·원문 찾기"' not in legacy_source
print("실습 6 통과: 대학 논문 메뉴를 별도로 배치하고 중복 expander를 제거했습니다.")